# Project 14 — Build Your First PyTorch MLP

## Project Description

**Objective**: Build an actual neural network using `torch.nn`.

Create a small multi-class classification model using a dataset of your choice.

Your project should:

1. Load or create a classification dataset.
1. Convert the features and labels into PyTorch tensors.
1. Build an MLP using `nn.Module`.
1. Include at least:
   - two hidden layers,
   - nonlinear activation functions,
   - one output layer.
1. Inspect:
   - model architecture,
   - parameter names,
   - parameter shapes,
   - total parameter count.
1. Move the model and data to an appropriate device.
1. Perform a forward pass.
1. Compute the loss using `CrossEntropyLoss`.
1. Perform one backward pass.
1. Verify that model parameters have gradients.
1. Do not build the complete training loop yet.


**Important constraint**

For this project, the goal is **model construction**, not achieving high accuracy.

We are deliberately stopping before building the complete training pipeline. That will be the subject of the next stage.

## My Solution

### Imports & Setup

First, we import `torch` for core `PyTorch` functionality and `torch.nn` for building neural network components like layers and loss functions. We also use `make_classification` from `sklearn` to generate synthetic data.

In [1]:
import torch
import torch.nn as nn
from sklearn.datasets import make_classification

### Step 1 & 2: Dataset Creation & Tensor Conversion

- **Dataset Creation**: We create 100 samples with 4 features each, split across 3 target classes.

- **Feature Tensors**: Neural networks perform floating-point matrix operations, so features must be `torch.float32`.

- **Label Tensors**: `nn.CrossEntropyLoss` expects class indices as 64-bit integers (`torch.long`), not floats or one-hot vectors.

In [2]:
# Create synthetic multi-class classification dataset
X_raw, y_raw = make_classification(
    n_samples=100, 
    n_features=4, 
    n_informative=4, 
    n_redundant=0, 
    n_classes=3, 
    random_state=42
)

# Convert to PyTorch tensors with correct data types
X = torch.tensor(X_raw, dtype=torch.float32)
y = torch.tensor(y_raw, dtype=torch.long)

print(f"Features shape: {X.shape} | Data type: {X.dtype}")
print(f"Labels shape:   {y.shape}   | Data type: {y.dtype}")

Features shape: torch.Size([100, 4]) | Data type: torch.float32
Labels shape:   torch.Size([100])   | Data type: torch.int64


### Step 3 & 4: Build the MLP Architecture

We define a custom class inheriting from nn.Module.

- **`__init__()`**: Defines two hidden layers using `nn.Linear` (4 $\rightarrow$ 8 $\rightarrow$ 8) followed by `nn.ReLU()` activations, and a final output layer (8 $\rightarrow$ 3).
- **`forward()`**: Defines how data flows through the layers.
- **No Softmax in Model**: PyTorch's `CrossEntropyLoss` combines **Softmax** and **negative log-likelihood loss** internally for numerical stability. The model must output raw, unnormalized scores (logits).

In [3]:
class MultiClassMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(MultiClassMLP, self).__init__()
        
        # Hidden Layer 1
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.relu1 = nn.ReLU()
        
        # Hidden Layer 2
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)
        self.relu2 = nn.ReLU()
        
        # Output Layer (outputs unnormalized logits)
        self.output_layer = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.relu1(self.layer1(x))
        x = self.relu2(self.layer2(x))
        out = self.output_layer(x)
        return out

# Instantiate model (4 inputs, 8 hidden units per layer, 3 output classes)
model = MultiClassMLP(input_dim=4, hidden_dim=8, output_dim=3)

### Step 5: Model Inspection

To verify our model setup, we inspect:

- **Structure**: The layer hierarchy defined in `__init__()`.
- **Parameters**: The weight matrices ($W$) and bias vectors ($b$) for each layer.
- **Parameter Count**: Total trainable parameters calculated using **.numel()** (number of elements).

In [6]:
print("=== Model Structure ===")
print(model)

print("\n=== Parameter Breakdown ===")
total_params = 0

for name, param in model.named_parameters():
    param_count = param.numel()
    total_params += param_count
    print(f"Name: {name:<20} | Shape: {str(list(param.shape)):<12} | Count: {param_count}")

print(f"\nTotal Model Parameter Count: {total_params}")

=== Model Structure ===
MultiClassMLP(
  (layer1): Linear(in_features=4, out_features=8, bias=True)
  (relu1): ReLU()
  (layer2): Linear(in_features=8, out_features=8, bias=True)
  (relu2): ReLU()
  (output_layer): Linear(in_features=8, out_features=3, bias=True)
)

=== Parameter Breakdown ===
Name: layer1.weight        | Shape: [8, 4]       | Count: 32
Name: layer1.bias          | Shape: [8]          | Count: 8
Name: layer2.weight        | Shape: [8, 8]       | Count: 64
Name: layer2.bias          | Shape: [8]          | Count: 8
Name: output_layer.weight  | Shape: [3, 8]       | Count: 24
Name: output_layer.bias    | Shape: [3]          | Count: 3

Total Model Parameter Count: 139


### Step 6: Hardware Accelerator Assignment

For PyTorch to perform computations, both the model and the data tensors must be on the exact same device (cpu or cuda).

In [7]:
# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Move model and data to the designated device
model = model.to(device)
X = X.to(device)
y = y.to(device)

Using device: cpu


### Step 7 & 8: Forward Pass & Loss Calculation

- **Forward Pass**: Passing `X` into `model(X)` executes the forward method, generating a batch of raw output predictions (logits) with shape `[100, 3]`.

- **Loss Computation**: `nn.CrossEntropyLoss` compares the unnormalized logits to the true class indices `y` and yields a single scalar loss value.

In [8]:
# Step 7: Forward Pass
logits = model(X)
print(f"Logits output shape: {logits.shape}")

# Step 8: Compute Loss
criterion = nn.CrossEntropyLoss()
loss = criterion(logits, y)
print(f"Calculated Loss Value: {loss.item():.4f}")

Logits output shape: torch.Size([100, 3])
Calculated Loss Value: 1.0936


### Step 9 & 10: Backward Pass & Gradient Verification

- **Backward Pass** (`loss.backward()`): PyTorch uses automatic differentiation (autograd) to compute gradients ($\frac{\partial \text{loss}}{\partial w}$) for all trainable parameters.
- **Gradient Verification**: After backpropagation, each weight and bias tensor stores its calculated gradient in its `.grad` attribute.

In [9]:
# Step 9: Backward Pass
loss.backward()

# Step 10: Verify Gradients
print("=== Gradient Verification ===")
for name, param in model.named_parameters():
    has_grad = param.grad is not None and torch.any(param.grad != 0)
    grad_norm = param.grad.norm().item() if param.grad is not None else 0.0
    print(f"Param: {name:<20} | Has Gradient: {has_grad!s:<5} | Grad Norm: {grad_norm:.6f}")

=== Gradient Verification ===
Param: layer1.weight        | Has Gradient: tensor(True) | Grad Norm: 0.053794
Param: layer1.bias          | Has Gradient: tensor(True) | Grad Norm: 0.017470
Param: layer2.weight        | Has Gradient: tensor(True) | Grad Norm: 0.070894
Param: layer2.bias          | Has Gradient: tensor(True) | Grad Norm: 0.034973
Param: output_layer.weight  | Has Gradient: tensor(True) | Grad Norm: 0.050524
Param: output_layer.bias    | Has Gradient: tensor(True) | Grad Norm: 0.102770


## What ChatGPT Expected Me to Learn from this Project 14

Project 14 was deliberately incomplete.

That was intentional.

You were expected to learn how to:

- construct a neural network,
- define its architecture,
- inspect its parameters,
- perform a forward pass,
- compute a loss,
- perform a backward pass.

The project stopped before repeated training because we hadn't yet studied the machinery that manages:

- batches,
- epochs,
- data loading,
- parameter updates,
- evaluation.

Now we put those pieces together.